# Registering Externally hosted ML Models to OpenSearch

Prerequisit
- [model_hosting](../../model_hosting/README.md)

### Install python modules

In [1]:
import sys
!{sys.executable} -m pip install opensearch-py

### Load helper modules

In [2]:
import time
import pprint

### Create an OpenSearch Client

Your opensearch password should be available in `~/.env`

```bash
    OPENSEARCH_INITIAL_ADMIN_PASSWORD="strong password"
```

In [3]:
import os
from dotenv import load_dotenv
from opensearchpy import OpenSearch

load_dotenv()
host = 'localhost'
port = 9200
password = os.getenv("OPENSEARCH_INITIAL_ADMIN_PASSWORD")

client = OpenSearch(
    hosts=[{"host": host, "port": port}],
    http_auth=("admin", password),
    http_compress=True,
    use_ssl=True,
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)
pprint.pprint(client.info())

{'cluster_name': 'docker-cluster',
 'cluster_uuid': 'qIJ28Ej2TCC6_LI88zZLlw',
 'name': '4db878c40bab',
 'tagline': 'The OpenSearch Project: https://opensearch.org/',
 'version': {'build_date': '2026-02-07T07:54:31.169913465Z',
             'build_hash': 'bbc94f0bdc3a759011e6529ecfe52840856f91a3',
             'build_snapshot': False,
             'build_type': 'tar',
             'distribution': 'opensearch',
             'lucene_version': '10.3.2',
             'minimum_index_compatibility_version': '2.0.0',
             'minimum_wire_compatibility_version': '2.19.0',
             'number': '3.5.0'}}


### Register ML models to OpenSearch

Enable ML model registration

In [ ]:
host_ip = "your-host-ip-address"

In [ ]:
cluster_settings = {
    "persistent": {
        "plugins.ml_commons.only_run_on_ml_node": "false",
        "plugins.ml_commons.model_access_control_enabled": "true",
        "plugins.ml_commons.model_auto_deploy.enable": "false",
        "plugins.ml_commons.allow_registering_model_via_url": "true",
        "plugins.ml_commons.connector_access_control_enabled": "true",
        "plugins.ml_commons.trusted_connector_endpoints_regex": [
            "^http://localhost:8000/.*$",
            f"^http://{host_ip}:8000/.*$",
            f"^http://{host_ip}:8001/.*$" # Add port numbers for multiple models
        ]
    }
}

try:
    response = client.cluster.put_settings(body=cluster_settings)
    pprint.pprint(response)
except Exception as e:
    print(f"Failed to update cluster settings: {e}")

Create connector

In [ ]:
def create_connector(client, name, endpoint, path):
    body = {
        "name": name,
        "version": 1,
        "protocol": "http",
        "parameters": {
            "endpoint": endpoint
        },

        # OpenSearch 3.5 does not like an empty credential object.
        # Your FastAPI service does not need to use this value.
        "credential": {
            "openAI_key": "dummy_value"
        },

        "actions": [
            {
                "action_type": "predict",
                "method": "POST",
                "url": f"http://${{parameters.endpoint}}{path}",
                "headers": {
                    "Authorization": "Bearer ${credential.openAI_key}",
                    "Content-Type": "application/json"
                },
                "request_body": "{ \"input\": ${parameters.input} }"
            }
        ]
    }

    try:
        response = client.transport.perform_request(
            "POST",
            "/_plugins/_ml/connectors/_create",
            body=body
        )
        pprint.pprint(response)

    except Exception as e:
        print(type(e))
        print(e)

        if hasattr(e, "info"):
            pprint.pprint(e.info)

    return response["connector_id"]

In [ ]:
# Sparse model connector
sparse_connector_id = create_connector(
    client=client,
    name="sparse",
    endpoint=f"{host_ip}:8000",
    path="/predict"
)

# Dense model connectors
e5_passage_connector_id = create_connector(
    client=client,
    name="e5-passages",
    endpoint=f"{host_ip}:8001",
    path="/embed/passages"
)

e5_query_connector_id = create_connector(
    client=client,
    name="e5-query",
    endpoint=f"{host_ip}:8001",
    path="/embed/query"
)


Register a new model group

In [ ]:
def register_model_group(client, name, description="A model group for local models"):
    """
    Registers a new model group in OpenSearch ML Commons.
    
    Args:
        client: The OpenSearch client instance.
        name (str): The name of the model group.
        description (str): A description of the model group.
        
    Returns:
        str: The ID of the registered model group, or None if failed.
    """
    model_group_body = {
        "name": name,
        "description": description
    }

    try:
        response = client.transport.perform_request(
            "POST",
            "/_plugins/_ml/model_groups/_register",
            body=model_group_body
        )
        pprint.pprint(response)
        
        model_group_id = response.get("model_group_id")
        if model_group_id:
            print(f"Captured Model Group ID: {model_group_id}")
            return model_group_id
        else:
            print("Warning: Model group registered but ID not found in response.")
            return None

    except Exception as e:
        pprint.pprint(e.info)  # Print detailed error informatfion if available
        return None

In [ ]:
local_model_group_id = register_model_group(client, "local_model_group")

Register a remote model to the model group

- https://docs.opensearch.org/latest/ml-commons-plugin/remote-models/index/

In [ ]:
def register_remote_model(
    client,
    model_group_id,
    connector_id,
    name,
    description=None,
):
    body = {
        "name": name,
        "function_name": "remote",
        "model_group_id": model_group_id,
        "description": description or name,
        "connector_id": connector_id,
    }

    response = client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_register",
        body=body,
    )

    pprint.pprint(response)

    return response["model_id"]

Register and deploy a sparse encoder model

In [ ]:
sparse_model_register_id = register_remote_model(
    client=client,
    model_group_id=local_model_group_id,
    connector_id=sparse_connector_id,
    name="opensearch-neural-sparse-encoding-multilingual-v1",
    description="Remote multilingual sparse encoder",
)

In [ ]:
def deploy_model(client, model_id, wait=True, timeout=300, poll_interval=5):
    """
    Deploy a remote model and optionally wait until deployment completes.

    Returns:
        model_id
    """

    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{model_id}/_deploy",
    )

    pprint.pprint(response)

    if not wait:
        return model_id

    start = time.time()

    while True:
        model_info = client.transport.perform_request(
            "GET",
            f"/_plugins/_ml/models/{model_id}",
        )

        state = model_info.get("model_state")

        print(f"model_state={state}")

        if state == "DEPLOYED":
            print(f"Model deployed: {model_id}")
            return model_id

        if state == "DEPLOY_FAILED":
            raise RuntimeError(
                f"Deployment failed:\n{pprint.pformat(model_info)}"
            )

        if time.time() - start > timeout:
            raise TimeoutError(
                f"Deployment timeout after {timeout} seconds"
            )

        time.sleep(poll_interval)

In [ ]:
sparse_model_deploy_id = deploy_model(client, sparse_model_register_id)

Testing sparse encoding

In [ ]:
def predict_model(client, model_id, texts):
    """
    Run inference against a deployed OpenSearch model.

    Parameters
    ----------
    client : OpenSearch client
    model_id : str
        Deployed model ID.
    texts : str | list[str]
        Input text(s).

    Returns
    -------
    dict
        OpenSearch prediction response.
    """

    if isinstance(texts, str):
        texts = [texts]

    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{model_id}/_predict",
        body={
            "parameters": {
                "input": texts
            }
        }
    )

    return response


In [ ]:
response = predict_model(
    client,
    sparse_model_deploy_id,
    "情報検索と生成AIについて説明してください。"
)
pprint.pprint(response)

Register and deploy a dense model

In [ ]:
e5_passage_register_id = register_remote_model(
    client=client,
    model_group_id=local_model_group_id,
    connector_id=e5_passage_connector_id,
    name="intfloat/multilingual-e5-large",
    description="Remote multilingual dense encoder",
)

In [ ]:
e5_passage_deploy_id = deploy_model(client, e5_passage_register_id)

Test the dense model

In [ ]:
response = predict_model(
    client,
    e5_passage_deploy_id,
    "情報検索と生成AIについて説明してください。"
)
pprint.pprint(response)

### List deployed models

Search ML Commons for models that are currently deployed and return their IDs.

- https://docs.opensearch.org/latest/ml-commons-plugin/api/model-apis/search-model/

In [4]:
def list_deployed_models(
    client,
    states=("DEPLOYED", "PARTIALLY_DEPLOYED"),
    size=100,
):
    """
    List models currently deployed in OpenSearch ML Commons, grouped by model group.

    Parameters
    ----------
    client : OpenSearch client
    states : tuple[str, ...]
        Model states to include. Defaults to DEPLOYED and PARTIALLY_DEPLOYED.
    size : int
        Maximum number of models to return.

    Returns
    -------
    dict[str, list[dict]]
        Mapping of model_group_id -> list of
        {"model_id", "name", "model_state"} entries.
    """
    body = {
        "query": {
            "terms": {
                "model_state": list(states)
            }
        },
        "_source": ["name", "model_state", "model_group_id"],
        "size": size,
    }

    response = client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_search",
        body=body,
    )

    grouped = {}
    for hit in response["hits"]["hits"]:
        source = hit["_source"]
        group_id = source.get("model_group_id", "<no group>")
        grouped.setdefault(group_id, []).append(
            {
                "model_id": hit["_id"],
                "name": source.get("name"),
                "model_state": source.get("model_state"),
            }
        )

    for group_id, models in grouped.items():
        print(f"Model group: {group_id}")
        for model in models:
            print(
                f"  - {model['model_id']} "
                f"({model['name']}, {model['model_state']})"
            )
        print()

    return grouped

In [19]:
deployed_models = list_deployed_models(client)

Model group: MjJAGJ8B32CkJZK9EMta
  - QTJOGJ8B32CkJZK9zMsG (opensearch-neural-sparse-encoding-multilingual-v1, DEPLOYED)
  - RjJSGJ8B32CkJZK9N8tW (intfloat/multilingual-e5-large, DEPLOYED)



### Delete models

Delete a single model, or every model under a model group. A model must be
undeployed before it can be deleted, so `delete_model` undeploys first.

- https://docs.opensearch.org/latest/ml-commons-plugin/api/model-apis/delete-model/

In [6]:
def undeploy_model(client, model_id, wait=True, timeout=300, poll_interval=5):
    """
    Undeploy a model from OpenSearch ML Commons and optionally wait until
    it is fully undeployed.

    Parameters
    ----------
    client : OpenSearch client
    model_id : str
        The model to undeploy.
    wait : bool
        Poll the model state until it is UNDEPLOYED.
    timeout : int
        Maximum seconds to wait when wait=True.
    poll_interval : int
        Seconds between state checks.

    Returns
    -------
    model_id
    """
    response = client.transport.perform_request(
        "POST",
        f"/_plugins/_ml/models/{model_id}/_undeploy",
    )

    pprint.pprint(response)

    if not wait:
        return model_id

    start = time.time()

    while True:
        model_info = client.transport.perform_request(
            "GET",
            f"/_plugins/_ml/models/{model_id}",
        )

        state = model_info.get("model_state")

        print(f"model_state={state}")

        if state == "UNDEPLOYED":
            print(f"Model undeployed: {model_id}")
            return model_id

        if time.time() - start > timeout:
            raise TimeoutError(
                f"Undeploy timeout after {timeout} seconds"
            )

        time.sleep(poll_interval)

In [7]:
def delete_model(client, model_id, undeploy=True):
    """
    Delete a model from OpenSearch ML Commons.

    A deployed model cannot be deleted, so it is undeployed first.

    Parameters
    ----------
    client : OpenSearch client
    model_id : str
        The model to delete.
    undeploy : bool
        Undeploy the model before deleting it.

    Returns
    -------
    dict
        The delete API response.
    """
    if undeploy:
        try:
            undeploy_model(client, model_id)
        except Exception as e:
            print(f"Undeploy skipped/failed for {model_id}: {e}")

    response = client.transport.perform_request(
        "DELETE",
        f"/_plugins/_ml/models/{model_id}",
    )

    pprint.pprint(response)

    return response

In [ ]:
delete_model(client, "model-id")

{'lB8dmVZnQG2RU5E3tysT4g': {'stats': {'NzJEGJ8B32CkJZK9pssd': 'undeployed'}}}
model_state=UNDEPLOYED
Model undeployed: NzJEGJ8B32CkJZK9pssd
{'_id': 'NzJEGJ8B32CkJZK9pssd',
 '_index': '.plugins-ml-model',
 '_primary_term': 2,
 '_seq_no': 2249,
 '_shards': {'failed': 0, 'successful': 1, 'total': 1},
 '_version': 9,
 'forced_refresh': True,
 'result': 'deleted'}


{'_index': '.plugins-ml-model',
 '_id': 'NzJEGJ8B32CkJZK9pssd',
 '_version': 9,
 'result': 'deleted',
 'forced_refresh': True,
 '_shards': {'total': 1, 'successful': 1, 'failed': 0},
 '_seq_no': 2249,
 '_primary_term': 2}

In [12]:
def delete_models_in_group(client, model_group_id, delete_group=False, size=1000):
    """
    Delete every model under a model group, and optionally the group itself.

    Parameters
    ----------
    client : OpenSearch client
    model_group_id : str
        The model group whose models should be deleted.
    delete_group : bool
        Also delete the (now empty) model group afterwards.
    size : int
        Maximum number of models to look up in the group.

    Returns
    -------
    list[str]
        The model IDs that were deleted.
    """
    body = {
        "query": {
            "term": {
                "model_group_id": model_group_id
            }
        },
        "_source": ["name", "model_state"],
        "size": size,
    }

    response = client.transport.perform_request(
        "POST",
        "/_plugins/_ml/models/_search",
        body=body,
    )

    model_ids = [hit["_id"] for hit in response["hits"]["hits"]]
    print(f"Found {len(model_ids)} model(s) in group {model_group_id}")

    for model_id in model_ids:
        print(f"Deleting model {model_id}")
        delete_model(client, model_id)

    if delete_group:
        group_response = client.transport.perform_request(
            "DELETE",
            f"/_plugins/_ml/model_groups/{model_group_id}",
        )
        pprint.pprint(group_response)

    return model_ids

In [ ]:
delete_models_in_group(client, "model-group-id", delete_group=True)